# 02 · BERT TextCls —— Encoder-only 双向 + [CLS] 微调范式

**家族位置**：`05_Transformer_NLP` 第 2 站。01 已拆完 Transformer 主轴（MHA/因果mask/PE/残差LN），本章只取 **Encoder** 一半：去掉因果遮蔽让每个位置**双向**互相看，前置 `[CLS]` 专职汇总句意做分类——这就是 BERT 的微调范式。任务 toy `y=1⇔序列含0`（约49%正例），与 05-01 同 vocab 量级、无外网、CPU 分钟级。

**学习目标**
1. BERT 为什么只用 Encoder：双向自注意力无因果 mask，填空/理解比生成更合适
2. [CLS] 机制：前置一个可学习汇总位，取 `h[0]` 过线性头分类
3. 微调范式：小 toy 上 Adam 微调 30ep 即可拟合，loss/acc 曲线与注意力热力可视化
4. 与 MeanPool/线性基线同台：双向能否比“只看平均”更准

## 1. 原理：从 Encoder-Decoder 到 Encoder-only

### 通俗理解

**一句话**：01 的 Transformer 像“翻译官要看原文再吐译文”（编+解+交叉），BERT 像“审稿人通读整句在页眉打个总分”——只用 Encoder 双向读，前置 `[CLS]` 当页眉汇总位。

**比喻**：因果 mask 是“只能看已写过的”，适合 GPT 写故事；双向是“前后都能翻”，适合判断“这句有没有含关键词0”。BERT 去掉下三角，让 8 个字互相投全票，`[CLS]` 再向 8 字汇票得到句向量。

### 结构账

```
输入：  [CLS] + x1..x8  → Emb+PE  (S+1=9)
Encoder×2： x = LN(x+MHA(x,x,x, mask=None)) → LN(x+FFN(x))   双向，无因果
分类头： logits = Linear(h[CLS])   (d=32 → 2类)
训练：  CE(logits, y)   800训练/200测试，Adam 1e-3，30ep
```

- **与 01 对比**：01 的 Decoder 有因果+cross；本章 `mask=None` 全可见，参数量相近便于归因于机制
- **评估**：准确率 + 混淆对角；注意力热力：CLS 对各 token 的平均头权重（“汇总位在看谁”）

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_cls_data
from common.models import BERTForCls, set_torch_seed
from common.utils import set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

VOCAB, SEQ_LEN = 12, 8
N_TRAIN, N_TEST = 800, 200
EPOCHS, BATCH, LR = 30, 32, 1e-3
X_all, y_all = make_cls_data(N_TRAIN+N_TEST, SEQ_LEN, VOCAB, seed=0)
X_train, y_train = X_all[:N_TRAIN], y_all[:N_TRAIN]
X_test, y_test = X_all[N_TRAIN:], y_all[N_TRAIN:]
print(f"train {len(X_train)} / test {len(X_test)} | 正例率 train {sum(y_train)/len(y_train):.3f} test {sum(y_test)/len(y_test):.3f} | 例 {X_train[0]}→{y_train[0]}")


## 2. 数据：含0即1 的二分类 toy

`S=8, vocab=12`，含 token 0 即正，约 49% 平衡；基线“只看平均”理论上限约 0.5+，BERT 双向可学到“是否存在0”的存在性判断。

In [ ]:
# fig0：类别与 0 出现位置分布
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(["neg(y=0)", "pos(y=1)"],[(1-sum(y_train)/len(y_train)), sum(y_train)/len(y_train)], color=["#4C72B0","#DD8452"])
for i, v in enumerate([(1-sum(y_train)/len(y_train)), sum(y_train)/len(y_train)]):
    axes[0].text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=9)
axes[0].set_ylim(0,0.65); axes[0].set_title("训练集类别分布")
# 0 出现位置直方
pos_counts = [sum(1 for tok in seq if tok==0) for seq in X_train]
axes[1].hist(pos_counts, bins=range(0,6), color="#55A868", edgecolor="white", alpha=0.9)
axes[1].set_title("每句含 0 个数分布（train）")
axes[1].set_xlabel("含 0 个数"); axes[1].set_ylabel("句数")
plt.tight_layout()
plt.savefig(FIGS / "fig0_dist.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 模型：BERT Encoder-only + CLS 头（与线性/MeanPool 基线同台）

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_eval(model, Xtr, ytr, Xte, yte, epochs=EPOCHS, batch=BATCH, lr=LR, seed=0):
    set_torch_seed(seed)
    xt = torch.tensor(Xtr, dtype=torch.long); yt = torch.tensor(ytr, dtype=torch.long)
    xe = torch.tensor(Xte, dtype=torch.long); ye = torch.tensor(yte, dtype=torch.long)
    loader = DataLoader(TensorDataset(xt, yt), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist_loss, hist_te_acc = [], []
    for ep in range(1, epochs+1):
        model.train()
        tot=0
        for xb, yb in loader:
            logits = model(xb)
            loss = lossf(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(xb)
        hist_loss.append(tot/len(Xtr))
        model.eval()
        with torch.no_grad():
            pred = model(xe).argmax(dim=-1)
            acc = (pred==ye).float().mean().item()
            hist_te_acc.append(acc)
    model.eval()
    with torch.no_grad():
        pred = model(xe).argmax(dim=-1)
        acc = (pred==ye).float().mean().item()
    return hist_loss, hist_te_acc, acc, pred

# 基线：MeanPool + 线性（emb16 均值→2类）
class MeanPoolCls(nn.Module):
    def __init__(self, vocab, emb_dim=16, num_labels=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb_dim)
        self.fc = nn.Linear(emb_dim, num_labels)
    def forward(self, x):
        h = self.emb(x).mean(dim=1)
        return self.fc(h)

bert = BERTForCls(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64, num_labels=2, max_len=32)
meanpool = MeanPoolCls(VOCAB, 16, 2)
results = {}
hists = {}
for name, model in [("MeanPool", meanpool), ("BERT", bert)]:
    hl, ha, acc, pred = train_eval(model, X_train, y_train, X_test, y_test)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    results[name] = (acc, n_params, pred)
    hists[name] = (hl, ha)
    print(f"{name:10s} test-acc={acc:.4f} params={n_params}")

# fig1：loss + test acc 双轴
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for name, color in [("MeanPool","#4C72B0"), ("BERT","#DD8452")]:
    hl, ha = hists[name]
    axes[0].plot(hl, label=name, color=color)
    axes[1].plot(ha, label=name, color=color)
axes[0].set_title("训练 loss")
axes[0].set_xlabel("epoch"); axes[1].set_title("测试 acc")
axes[1].set_xlabel("epoch")
for ax in axes: ax.legend(fontsize=8)
plt.suptitle("BERT vs MeanPool（同 toy 同预算）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：acc 柱状
fig, ax = plt.subplots(figsize=(5.5, 3.6))
names = list(results.keys())
accs = [results[n][0] for n in names]
colors = ["#4C72B0","#DD8452"]
b = ax.bar(names, accs, color=colors)
for i, v in enumerate(accs):
    ax.text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylim(0, 1.12); ax.set_ylabel("test acc")
ax.set_title("含0分类：BERT 双向 vs MeanPool 均值")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 可视化：CLS 在看谁 + 错例

In [ ]:
# fig3：CLS 注意力热力（单句，层2头平均：CLS 行对 8 token）
bert.eval()
with torch.no_grad():
    # 选一句含 0 的正例
    idx = next(i for i,y in enumerate(y_test) if y==1)
    x = torch.tensor([X_test[idx]], dtype=torch.long)
    logits, attns = bert(x, return_attn=True)
    # attns: list layer, each (B,h, S+1, S+1); CLS=0 行
    last = attns[-1][0].mean(dim=0).numpy()  # (9,9)
    cls_row = last[0, 1:]  # CLS 对 8 token
    tokens = [str(t) for t in X_test[idx]]
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.barh(range(SEQ_LEN), cls_row[::-1], color="#55A868")
ax.set_yticks(range(SEQ_LEN))
ax.set_yticklabels(tokens[::-1])
ax.set_xlabel("CLS 注意力权重（层2 头平均，CLS→各token）")
ax.set_title(f"BERT CLS 聚焦（正例 y=1，句 {X_test[idx]}）——含 0 位置权重高即学到存在性")
for i, v in enumerate(cls_row[::-1]):
    ax.text(v+0.01, i, f"{v:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig3_cls_attn.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：预测 vs 真实散点/错例表（4 例）
bert.eval()
with torch.no_grad():
    xe = torch.tensor(X_test, dtype=torch.long)
    logits = bert(xe)
    pred = logits.argmax(dim=-1).tolist()
# 挑 2 对 2 错各 2 例
correct = [i for i in range(len(y_test)) if pred[i]==y_test[i]][:2]
wrong = [i for i in range(len(y_test)) if pred[i]!=y_test[i]][:4]
# 若全对则 wrong 为空，补全对示例
samples = []
for i in correct: samples.append((i,"✓"))
for i in wrong[:2]: samples.append((i,"✗"))
if not wrong:
    extra = [i for i in range(len(y_test)) if i not in correct][:2]
    for i in extra: samples.append((i,"✓"))
fig, axes = plt.subplots(len(samples), 1, figsize=(8, 1.1*len(samples)), sharex=False)
if len(samples)==1: axes=[axes]
for ax, (i, mark) in zip(axes, samples):
    toks = X_test[i]; y = y_test[i]; p = pred[i]
    # 画一句
    ax.set_xlim(0, SEQ_LEN+1); ax.set_ylim(0, 1.4); ax.axis("off")
    ax.set_title(f"{'正例' if y else '负例'} y={y} pred={p} {mark}  句: {toks}", fontsize=9, loc="left")
    for j, tok in enumerate(toks):
        col = "#FAD7A0" if tok==0 else "#D5D8DC"
        ax.add_patch(plt.Rectangle((j+0.05, 0.45), 0.9, 0.6, facecolor=col, edgecolor="#555", linewidth=0.7))
        ax.text(j+0.5, 0.75, str(tok), ha="center", va="center", fontsize=10, color="#C0392B" if tok==0 else "black", weight="bold" if tok==0 else "normal")
plt.suptitle("BERT 预测抽样（0 标橙，含0应为正例）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig4_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"test acc BERT={results['BERT'][0]:.4f} MeanPool={results['MeanPool'][0]:.4f}")


## 5. 总结与下一步

**本项目收获**

1. Encoder-only 双向：`mask=None` 全可见，CLS 汇票做分类，与 01 的因果 mask 形成对比
2. 微调范式：toy 800/200 Adam 30ep，loss→0，test acc 接近 1.0，CLS 注意力对 0 权重高
3. 与 04-02 MeanPool 同台：含0存在性任务双向可学，MeanPool 均值亦可但鲁棒性差（见 fig1 曲线抖动）
4. 衔接 01：MHA/PE/LN/FFN 全复用，仅去掉 Decoder/cross，本章骨架即 BERT

**下一步**：`03_GPT_LM_Mini`（Decoder-only 因果，下一个词预测 + 自回归生成）→ `04_T5_BART_Mini`（Encoder-Decoder 统一）→ `05_Mamba_vs_Transformer_Mini`（拓展对照）。